# OSR-303 Electromagnetic Fault Injection Example

## Connection Diagram

Use [powershorter](https://github.com/OSR-Lab/powershorter) to drive the EMPulser for electromagnetic fault injection on the OSR-303 board, and use a PICO3206D oscilloscope to observe the glitch.

<img src="images/emfi-303.jpg"  width="800">

<img src="images/emfi.png"  width="800">

**Electromagnetic pulse disturbance can prevent the 303 board from operating normally. We use the PowerShorter's relay to perform a hard reset of the board.**

**To enable the reset, switch the 303 board's power supply to external during electromagnetic fault injection. Route the external 3.3V voltage through the PowerShorter relay, then connect it to the board's external power input (the external 3.3V can also be taken from the 3.3V on the 303 board's header).**

<p style="color:#FF0000">  ⚡<b>The EMPulser is a high-voltage device. Do not touch the electromagnetic pulse output probe while it is operating. Mounting and removing the EM fault-injection probe must be done while the device is powered off!</b> ⚡</p>

## Download the Project

As shown in `https://github.com/OSR-Lab/osr-303/blob/main/project/STM32CubeIDE-303Guide.md`, we use STM32CubeIDE to download the `FORLOOP` project (in the project folder).

## Communication Test

First set the power switch to internal, then press the reset button once to perform a communication test.

In [1]:
import serial

In [2]:
loopv = 200

In [3]:
toe = serial.Serial('com50', 115200, timeout=1)

In [4]:
toe.write(loopv.to_bytes(1, 'little'))
ret = toe.read(1)
retv = int.from_bytes(ret, 'little')
print(retv)

200


## Controlling the 303 Reset

To control the reset during electromagnetic pulsing, use the `relay` of `powershorter` to perform a hard reset of the chip; set the power switch to external supply at this point.

In [5]:
import power_shorter as ps
import time

In [6]:
em_dev = ps.EMPulser('com6') # select the serial port

In [7]:
def reset_toe(): 
    em_dev.relay(ps.RELAY.RELAY2, 0)
    time.sleep(0.3)
    em_dev.relay(ps.RELAY.RELAY2, 1)
    time.sleep(1)
    toe.reset_input_buffer()

In [8]:
# Test the reset functionality
reset_toe()  
toe.write(loopv.to_bytes(1, 'little'))
ret = toe.read(1)
retv = int.from_bytes(ret, 'little')
print(retv)

200


## Electromagnetic Fault Injection

In [9]:
def glitch(delay, pulse):
    em_dev.engine_cfg(ps.Engine.E1, delay, pulse, trigger_mode=ps.TRIGGER_MODE.RISE, trigger_edges=1) # on receiving the trigger signal, wait delay*10 ns, then generate an EM pulse for pulse*10 ns
    em_dev.arm(ps.Engine.E1)
    toe.write(loopv.to_bytes(1, 'little'))
    ret = toe.read(1)
    retv = int.from_bytes(ret, 'little')
    state = None
    if ret == b'':
        state  = 'dead'
        reset_toe()
    elif retv == loopv:
        state = 'normal'
    else:
        state = 'glitch success'
        #print(state, retv, delay, pulse)
    return state, retv, delay, pulse

In [10]:
glitch(3200, 10)

('normal', 200, 3200, 10)

The pico oscilloscope observes the trigger signal output by the board, and you can observe a glitch - this glitch is the electromagnetic pulse disturbing the chip's internal circuitry.
![image](images/em-glitch-pico.png)

## Visualizing Fault Parameters and Results

You can conveniently observe fault parameters and results using [FaultViz](https://github.com/OSR-Lab/faultviz).

In [11]:
import faultviz

In [ ]:
faultviz.start_view_service()

In [13]:
vt = faultviz.ViewWidget()

In [14]:
state, retv, delay, pulse = glitch(3200, 20)
vt.update(state=state, val=retv, delay=delay, pulse=pulse)

In [ ]:
vt.show()

## Random-Parameter Injection

In [16]:
from tqdm.notebook import tnrange
import random

During electromagnetic fault injection, the positions of the EM injection probe and the chip are shown in the figure. By adjusting the EMPulser's energy knob, you can set the EM injection probe voltage to around 200V.

Depending on the specific environment, the position and voltage energy may need fine-tuning.

<img src="images/emfi-dir.jpg"  width="500">

In [17]:
for i in tnrange(1000):
    delay=random.randint(2000, 4000)
    pulse=random.randint(1, 3)
    state, retv, delay, pulse = glitch(delay, pulse)
    vt.update(state=state, val=retv, delay=delay, pulse=pulse)

  0%|          | 0/1000 [00:00<?, ?it/s]

## Results

![image](images/em-faultviz.png)